# Tugas 4 Bagian A: Audit Data Geospasial

Audit tanpa peta. Kolom teks aduan tidak dibaca di notebook ini.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 200, "display.max_columns", 30)
DATA = Path("../../week9/datasets")
HASIL = Path("../datasets")
HASIL.mkdir(exist_ok=True)

kolom = ["id_aduan", "tanggal", "kanal", "kelurahan", "lon", "lat"]
a = pd.read_csv(DATA / "203012420009_aduan.csv", usecols=kolom, dtype={"id_aduan": str})
w = pd.read_csv(DATA / "wilayah_kota_sukaraya.csv").set_index("kelurahan")
b = pd.read_csv(DATA / "aduan_bersih.csv", usecols=["id_aduan", "tanggal", "n_kirim"], dtype={"id_aduan": str})
b = b.rename(columns={"tanggal": "tanggal_baku"})

a["unik"] = a.id_aduan.isin(b.id_aduan)
print("baris mentah", len(a), "| id unik", a.id_aduan.nunique(), "| korpus bersih", a.unik.sum())

baris mentah 5010 | id unik 5010 | korpus bersih 4785


## 1. Kosong dan nol

In [2]:
kosong = a.lon.isna() | a.lat.isna()
nol = (a.lon == 0) & (a.lat == 0)
print("kosong", kosong.sum(), "| nol-nol", nol.sum(), "| satu kolom nol", ((a.lon == 0) ^ (a.lat == 0)).sum())
print("di korpus bersih: kosong", (kosong & a.unik).sum(), "| nol-nol", (nol & a.unik).sum())

kosong 55 | nol-nol 20 | satu kolom nol 0
di korpus bersih: kosong 53 | nol-nol 20


## 2. Urutan: batas peta sebelum dan sesudah nol dibuang

In [3]:
X0, X1, Y0, Y1 = 107.56, 107.64, -6.934, -6.88
tukar = a.lon.between(Y0, Y1) & a.lat.between(X0, X1)
lon_baik, lat_baik = a.lon.where(~tukar, a.lat), a.lat.where(~tukar, a.lon)
batas = lambda x, y: tuple(round(float(v), 4) for v in (x.min(), x.max(), y.min(), y.max()))
print("data mentah                  :", batas(a.lon, a.lat))
print("tertukar diperbaiki, nol ada :", batas(lon_baik, lat_baik))
print("tertukar diperbaiki, nol buang:", batas(lon_baik[~nol], lat_baik[~nol]))

data mentah                  : (-6.9332, 107.64, -6.934, 107.6389)
tertukar diperbaiki, nol ada : (0.0, 107.64, -6.934, 0.0)
tertukar diperbaiki, nol buang: (107.56, 107.64, -6.934, -6.88)


## 3. Di luar kota dan pola asalnya

In [4]:
a["x"], a["y"] = lon_baik, lat_baik
a.loc[nol, ["x", "y"]] = np.nan

def cari_kelurahan(x, y):
    if pd.isna(x):
        return None
    for nama, r in w.iterrows():
        if abs(x - r.lon_pusat) <= .01 and abs(y - r.lat_pusat) <= .009:
            return nama
    return None

a["kel_koordinat"] = [cari_kelurahan(x, y) for x, y in zip(a.x, a.y)]
luar_awal = a.lon.notna() & ~nol & ~(a.lon.between(X0, X1) & a.lat.between(Y0, Y1))
print("di luar kota", luar_awal.sum(), "| pola tertukar", (luar_awal & tukar).sum())
print("sesudah ditukar: di luar kota", (a.x.notna() & a.kel_koordinat.isna()).sum(),
      "| cocok label", (a.kel_koordinat[tukar] == a.kelurahan[tukar]).sum(), "dari", tukar.sum())
print("batas sesudah diperbaiki:", (a.x.min(), a.x.max(), a.y.min(), a.y.max()))
print(a.kanal[tukar].value_counts().to_dict())

di luar kota 30 | pola tertukar 30
sesudah ditukar: di luar kota 0 | cocok label 30 dari 30
batas sesudah diperbaiki: (np.float64(107.560008), np.float64(107.639982), np.float64(-6.933996), np.float64(-6.880014))
{'Aplikasi Mobile': 11, 'Web': 10, 'WhatsApp': 5, 'Datang Langsung': 3, 'Surat': 1}


## 4. Tumpukan di titik pusat kelurahan

In [5]:
pusat = {(r.lon_pusat, r.lat_pusat) for _, r in w.iterrows()}
a["di_pusat"] = [(x, y) in pusat for x, y in zip(a.x.round(6), a.y.round(6))]
ulang = a[a.x.notna()].groupby(["x", "y"]).size()
ulang = ulang[ulang > 1]
print("di titik pusat", a.di_pusat.sum(), "| korpus bersih", (a.di_pusat & a.unik).sum(),
      "| cocok label", (a.kel_koordinat[a.di_pusat] == a.kelurahan[a.di_pusat]).sum())
print("koordinat berulang di luar titik pusat:",
      sum((round(x, 6), round(y, 6)) not in pusat for x, y in ulang.index))
a[a.di_pusat].kelurahan.value_counts()

di titik pusat 75 | korpus bersih 72 | cocok label 75
koordinat berulang di luar titik pusat: 0


kelurahan
Babakan Sari       15
Rancabolang         9
Cibuntu Kaler       8
Kebon Gedang        6
Sekejati            6
Pasir Wangi         6
Cisaranten          5
Antapani Wetan      5
Cikawao             5
Margahayu Kidul     4
Mekarjaya           3
Ciburial            3
Name: count, dtype: int64

## 5. Label kelurahan dan koordinat

In [6]:
d = a[a.kel_koordinat.notna()]
beda = d.kel_koordinat != d.kelurahan
print("berselisih", beda.sum(), "dari", len(d), f"({beda.mean():.1%})")
print(d[beda].groupby(["kelurahan", "kel_koordinat"]).size().sort_values(ascending=False).to_string())
jarak = (d.y[beda] + 6.898).abs() * 111000
print("jarak ke garis lat -6.898 (m):", jarak.quantile([.5, .9]).round(0).to_dict())
pita = d[d.y.between(-6.898, -6.8967)]
print("titik di pita", len(pita), "| berlabel kelurahan selatan", (pita.kel_koordinat != pita.kelurahan).sum())

berselisih 102 dari 4935 (2.1%)
kelurahan       kel_koordinat  
Cibuntu Kaler   Pasir Wangi        27
Antapani Wetan  Mekarjaya          26
Sekejati        Rancabolang        26
Kebon Gedang    Babakan Sari       22
Cisaranten      Margahayu Kidul     1
jarak ke garis lat -6.898 (m): {0.5: 44.0, 0.9: 104.0}
titik di pita 267 | berlabel kelurahan selatan 101


## 6. Keputusan penanganan koordinat

In [7]:
# Kelurahan analisis: koordinat bila valid, selain itu label
a["kel_final"] = a.kel_koordinat.fillna(a.kelurahan)
a["sumber_kel"] = np.where(a.kel_koordinat.notna(), "koordinat", "label")
# Peta titik: tanpa koordinat kosong, nol, dan titik pusat
a["untuk_titik"] = a.x.notna() & a.kel_koordinat.notna() & ~a.di_pusat

u = a[a.unik].merge(b, on="id_aduan", how="left")
print("korpus", len(u), "| sumber", u.sumber_kel.value_counts().to_dict(), "| untuk peta titik", u.untuk_titik.sum())

korpus 4785 | sumber {'koordinat': 4712, 'label': 73} | untuk peta titik 4640


## 7. Urutan: hapus baris sebelum dan sesudah hitung per kelurahan

In [8]:
urut = pd.DataFrame({
    "semua_baris": a.kel_final.value_counts(),
    "aduan_unik": u.kel_final.value_counts(),
    "buang_kosong_nol_dulu": u[u.x.notna()].kel_final.value_counts(),
})
urut["hilang_jika_dibuang_dulu"] = urut.aduan_unik - urut.buang_kosong_nol_dulu
urut.sort_values("aduan_unik", ascending=False)

,semua_baris,aduan_unik,buang_kosong_nol_dulu,hilang_jika_dibuang_dulu
kel_final,,,,
Babakan Sari,860,819,810,9
Kebon Gedang,618,596,587,9
Mekarjaya,564,534,526,8
Rancabolang,483,463,456,7
Sekejati,467,450,443,7
Cibuntu Kaler,438,415,409,6
Cikawao,346,330,325,5
Antapani Wetan,344,330,325,5
Cisaranten,276,262,255,7


## 8. Perbandingan penyebut

In [9]:
t = pd.DataFrame({"penduduk": w.jumlah_penduduk, "luas_km2": w.luas_km2,
                  "aduan_label": u.kelurahan.value_counts(), "aduan": u.kel_final.value_counts()})
t["per_1000"] = (t.aduan / t.penduduk * 1000).round(2)
t["per_km2"] = (t.aduan / t.luas_km2).round(1)
t["rank_mentah"] = t.aduan.rank(ascending=False).astype(int)
t["rank_1000"] = t.per_1000.rank(ascending=False).astype(int)
t["rank_km2"] = t.per_km2.rank(ascending=False).astype(int)
spearman = lambda p, q: round(p.rank().corr(q.rank()), 3)
print("luas unik:", t.luas_km2.unique(), "| Spearman mentah~per km2:", spearman(t.aduan, t.per_km2),
      "| mentah~per 1000:", spearman(t.aduan, t.per_1000))
t.sort_values("per_1000", ascending=False)

luas unik: [4.42] | Spearman mentah~per km2: 1.0 | mentah~per 1000: 0.413


,penduduk,luas_km2,aduan_label,aduan,per_1000,per_km2,rank_mentah,rank_1000,rank_km2
Kebon Gedang,18300,4.42,617,596,32.57,134.8,2,1,2
Babakan Sari,62400,4.42,798,819,13.12,185.3,1,2,1
Ciburial,15600,4.42,195,195,12.50,44.1,11,3,11
Cibuntu Kaler,33800,4.42,440,415,12.28,93.9,6,4,6
Cisaranten,21700,4.42,263,262,12.07,59.3,9,5,9
Cikawao,27500,4.42,330,330,12.00,74.7,7,6,7
Sekejati,38600,4.42,476,450,11.66,101.8,5,7,5
Antapani Wetan,30200,4.42,354,330,10.93,74.7,7,8,7
Rancabolang,44900,4.42,437,463,10.31,104.8,4,9,4
Mekarjaya,52100,4.42,510,534,10.25,120.8,3,10,3


**Keputusan penyebut: aduan per km².** Luas wilayah menjadi dasar pembagian petugas dan armada, sehingga kelurahan yang kekurangan tenaga dapat dibantu dari area terdekat.

Catatan: luas semua kelurahan sama (4,42 km²), sehingga peringkat per km² sama dengan peringkat jumlah mentah. Per 1.000 penduduk dipakai sebagai pembanding.

## 9. Komposisi kanal per kelurahan

In [10]:
digital = u.kanal.isin(["Web", "Aplikasi Mobile"])
k = pd.DataFrame({
    "pct_digital": digital.groupby(u.kel_final).mean().mul(100).round(1),
    "pct_whatsapp": (u.kanal == "WhatsApp").groupby(u.kel_final).mean().mul(100).round(1),
    "digital_per_1000": (digital.groupby(u.kel_final).sum() / w.jumlah_penduduk * 1000).round(2),
    "nondigital_per_1000": ((~digital).groupby(u.kel_final).sum() / w.jumlah_penduduk * 1000).round(2),
})
k.sort_values("pct_digital")

,pct_digital,pct_whatsapp,digital_per_1000,nondigital_per_1000
Pasir Wangi,25.5,43.8,0.95,2.77
Cisaranten,59.9,16.8,7.24,4.84
Margahayu Kidul,60.9,16.4,5.85,3.75
Ciburial,65.6,15.4,8.21,4.29
Kebon Gedang,66.1,14.4,21.53,11.04
Cibuntu Kaler,66.3,18.8,8.14,4.14
Rancabolang,67.2,17.3,6.93,3.39
Antapani Wetan,68.8,13.3,7.52,3.41
Mekarjaya,70.4,14.4,7.22,3.03
Sekejati,71.1,14.7,8.29,3.37


## 10. Simpan untuk notebook peta

In [11]:
simpan = ["id_aduan", "tanggal_baku", "kanal", "kelurahan", "kel_final", "sumber_kel", "x", "y", "di_pusat", "untuk_titik", "n_kirim"]
u[simpan].to_csv(HASIL / "aduan_geo.csv", index=False)
print("tersimpan", len(u), "baris")

tersimpan 4785 baris
